In [1]:
import cupy as cp

In [60]:
import numba

In [2]:
import pycuda

In [3]:
import torch

In [4]:
if torch.cuda.is_available():
    device_properties = torch.cuda.get_device_properties(0)
    print(f"Device Name: {device_properties.name}")
    print(f"Total Memory: {device_properties.total_memory / (1024**3):.2f} GB")
    print(f"CUDA Capability: {device_properties.major}.{device_properties.minor}")
else:
    print("CUDA is not available.")

Device Name: NVIDIA GeForce RTX 4090
Total Memory: 23.99 GB
CUDA Capability: 8.9


In [10]:
from minio import Minio
import os
import pandas as pd
import platform
import time

In [9]:
os.name

'posix'

In [10]:
print(platform.system(), platform.release())

Linux 6.6.87.2-microsoft-standard-WSL2


In [2]:
os.environ["SSL_CERT_FILE"] = 'all-data/public.crt'

In [3]:
def download_data_without_creds(data):
    client = Minio(
    '94.124.179.195:9000',
    secure=True
    )
    data = client.fget_object(
        'datasets',
        f'data/{data}',
        f'all-data/{data}'
    )

In [6]:
download_data_without_creds('OpenML-har-orig-1.csv')

In [7]:
download_data_without_creds('OpenML-usps-orig-1.csv')

In [11]:
# GaMAC Test

In [8]:
from gamac.autoclustering import Gamac
from gamac.estimation.internal import Internal

In [4]:
def gamac_test(data, target_measures):
    measures = {"BR": Internal.BR, "OS": Internal.OS, "MCR": Internal.MCR, "SYM": Internal.SYM}
    used_measures = [measures[x] for x in target_measures.split(sep=',')]
    df = pd.read_csv(f'all-data/{data}')
    if 'class' in df.columns:
        classes = df['class'].tolist()
        # classes = [int(c[1]) for c in classes]
    else:
        classes = None
    df = df.drop('class', errors='ignore', axis=1)
    print(f'used data: {data}')
    print(f'used measures: {used_measures}')
    result = Gamac(target_measures=tuple(used_measures)).run(table=df, text=None, image=None, classes=classes)
    print(f'optimal.model: {result.model}')
    print(f'clusters: {result.model.labels_}')
    print(f'F1 score: {result.f1_score}')

In [17]:
start = time.time()
gamac_test('OpenML-har-orig.csv', 'SYM')
print('Work time:', time.time() - start)

used data: OpenML-har-orig.csv
used measures: [<Internal.SYM: ('sym', <function sym at 0x7f0017459c60>)>]


/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 0.44759058952331543s, {'SYM': 2.3566619348116e-07} ===
=== MEASURES 0.33731627464294434s, {'SYM': 0.0003814715373599406} ===
=== ALGO 2.6456451416015625s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 15, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 38.4997353553772s, FailedRun, {'bandwidth': 0.4505824806907181, 'max_iter': 294, 'tol': 2.6112746081016713e-05} ===
=== ALGO 5.558532238006592s, FailedRun, {'name': 'DBSCAN', 'eps': 0.6907286646112676, 'eps_sq': 0.47710608811566496, 'min_samples': 10} ===
=== ALGO 21.367307424545288s, FailedRun, {'min_cluster_size': 8, 'min_samples': 10} ===
=== MEASURES 0.24439668655395508s, {'SYM': 5.385651731288114e-05} ===
=== ALGO 36.13166308403015s, SuccessRun, {'threshold': 0.6952076084171146, 'branching_factor': 32, 'n_clusters': 7} ===
=== MEASURES 0.23161649703979492s, {'SYM': 0.0001628135804306658} ===
=== ALGO 0.3363487720489502s, SuccessRun, {'name': 'KMeans', 'n_clusters': 7, 'max_iter': 100, 'tol': 0.000

In [16]:
start = time.time()
gamac_test('OpenML-usps-orig.csv', 'OS')
print('Work time:', time.time() - start)

used data: OpenML-usps-orig.csv
used measures: [<Internal.OS: ('os', <function os at 0x7f001745b600>)>]


/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 0.004419565200805664s, {'OS': -11874.843649351544} ===
=== MEASURES 0.002147197723388672s, {'OS': -145.42566805747447} ===
=== ALGO 0.226912260055542s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 4, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 28.61929154396057s, FailedRun, {'bandwidth': 0.48027685436394224, 'max_iter': 104, 'tol': 4.513097774718048e-05} ===
=== ALGO 4.8600404262542725s, FailedRun, {'name': 'DBSCAN', 'eps': 0.35034741100116923, 'eps_sq': 0.1227433083952222, 'min_samples': 9} ===
=== ALGO 24.688478708267212s, FailedRun, {'min_cluster_size': 12, 'min_samples': 13} ===
=== MEASURES 0.010626554489135742s, {'OS': -56.866838101566266} ===
=== ALGO 62.66710925102234s, SuccessRun, {'threshold': 0.2228057754309866, 'branching_factor': 73, 'n_clusters': 9} ===
=== MEASURES 0.004667520523071289s, {'OS': -42.439932119747056} ===
=== ALGO 0.4781327247619629s, SuccessRun, {'name': 'KMeans', 'n_clusters': 12, 'max_iter': 100, 'tol': 0.0001, 'r

In [1]:
# test profiling

In [3]:
import cProfile

In [11]:
cProfile.run("gamac_test('OpenML-usps-orig.csv', 'OS')")

used data: OpenML-usps-orig.csv
used measures: [<Internal.OS: ('os', <function os at 0x71533a0b27a0>)>]


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 0.7615294456481934s, {'OS': -15119.216443254567} ===
=== MEASURES 0.007898569107055664s, {'OS': -41.9287711084196} ===
=== ALGO 5.799864292144775s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 13, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 34.71236300468445s, FailedRun, {'bandwidth': 0.4357225303769598, 'max_iter': 115, 'tol': 3.5960867042149694e-05} ===
=== ALGO 5.933248996734619s, FailedRun, {'name': 'DBSCAN', 'eps': 0.4401605389252784, 'eps_sq': 0.19374130002699153, 'min_samples': 6} ===
=== ALGO 4.739547252655029s, FailedRun, {'min_cluster_size': 15, 'min_samples': 7} ===
=== MEASURES 0.0025098323822021484s, {'OS': -52.39267809934716} ===
=== ALGO 42.94095730781555s, SuccessRun, {'threshold': 0.2960702868178686, 'branching_factor': 38, 'n_clusters': 11} ===
=== MEASURES 0.002294301986694336s, {'OS': -41.92924478500858} ===
=== ALGO 0.3157517910003662s, SuccessRun, {'name': 'KMeans', 'n_clusters': 11, 'max_iter': 100, 'tol': 0.0001, 'random_

In [19]:
import timeit
setup = """
from gamac.autoclustering import Gamac
from gamac.estimation.internal import Internal
from minio import Minio
import os
import pandas as pd
import platform
import time

def gamac_test(data, target_measures):
    measures = {"BR": Internal.BR, "OS": Internal.OS, "MCR": Internal.MCR, "SYM": Internal.SYM}
    used_measures = [measures[x] for x in target_measures.split(sep=',')]
    df = pd.read_csv(f'all-data/{data}')
    if 'class' in df.columns:
        classes = df['class'].tolist()
        # classes = [int(c[1]) for c in classes]
    else:
        classes = None
    df = df.drop('class', errors='ignore', axis=1)
    print(f'used data: {data}')
    print(f'used measures: {used_measures}')
    result = Gamac(target_measures=tuple(used_measures)).run(table=df, text=None, image=None, classes=classes)
    print(f'optimal.model: {result.model}')
    print(f'clusters: {result.model.labels_}')
    print(f'F1 score: {result.f1_score}')

"""

In [22]:
timeit.repeat(stmt="gamac_test('OpenML-waveform-5000-orig.csv', 'OS')", setup=setup, \
              repeat=5, \
              number=10, globals=None)

used data: OpenML-waveform-5000-orig.csv
used measures: [<Internal.OS: ('os', <function os at 0x71533a0b27a0>)>]


/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 0.0029327869415283203s, {'OS': -14525.588217650746} ===
=== MEASURES 0.0030355453491210938s, {'OS': -134.20272645624334} ===
=== ALGO 3.8741376399993896s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 15, 'max_iter': 100, 'init': 'k-means++', 'tol': 0.0001} ===
=== ALGO 10.952723264694214s, FailedRun, {'bandwidth': 0.35471110067403067, 'max_iter': 279, 'tol': 6.412336439923932e-05} ===
=== ALGO 2.7333199977874756s, FailedRun, {'name': 'DBSCAN', 'eps': 0.7373336345148911, 'eps_sq': 0.543660888586939, 'min_samples': 14} ===
=== ALGO 12.575343608856201s, FailedRun, {'min_cluster_size': 12, 'min_samples': 12} ===
=== MEASURES 0.0018687248229980469s, {'OS': -222.75361506002952} ===
=== ALGO 18.41570019721985s, SuccessRun, {'threshold': 0.1782288496338066, 'branching_factor': 33, 'n_clusters': 13} ===
=== MEASURES 0.001352548599243164s, {'OS': -122.347466276696} ===
=== ALGO 0.29450368881225586s, SuccessRun, {'name': 'KMeans', 'n_clusters': 14, 'max_iter': 100, 'tol': 0.

/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 0.0024471282958984375s, {'OS': -15432.772919527208} ===
=== MEASURES 0.0021867752075195312s, {'OS': -332.5147697434831} ===
=== ALGO 0.09932279586791992s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 3, 'max_iter': 100, 'init': 'k-means++', 'tol': 0.0001} ===
=== ALGO 11.82685923576355s, FailedRun, {'bandwidth': 0.3089205812778765, 'max_iter': 214, 'tol': 9.722874239390266e-05} ===
=== ALGO 2.6006438732147217s, FailedRun, {'name': 'DBSCAN', 'eps': 1.209218353476674, 'eps_sq': 1.4622090263848386, 'min_samples': 6} ===
=== ALGO 16.153236627578735s, FailedRun, {'min_cluster_size': 10, 'min_samples': 15} ===
=== MEASURES 0.0043909549713134766s, {'OS': -186.45518071471668} ===
=== ALGO 23.022964000701904s, SuccessRun, {'threshold': 0.25555295847500165, 'branching_factor': 43, 'n_clusters': 12} ===
=== MEASURES 0.0037848949432373047s, {'OS': -189.24204232168304} ===
=== ALGO 0.20062899589538574s, SuccessRun, {'name': 'KMeans', 'n_clusters': 8, 'max_iter': 100, 'tol': 0.

...

/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 0.0024733543395996094s, {'OS': -14025.5324997951} ===
=== MEASURES 0.0015842914581298828s, {'OS': -253.03530025045882} ===
=== ALGO 0.5901250839233398s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 6, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 12.108051061630249s, FailedRun, {'bandwidth': 0.26283844724422434, 'max_iter': 274, 'tol': 8.978980693637908e-05} ===
=== ALGO 2.6680831909179688s, FailedRun, {'name': 'DBSCAN', 'eps': 0.06601560838133884, 'eps_sq': 0.004358060549958295, 'min_samples': 7} ===
=== ALGO 13.865231275558472s, FailedRun, {'min_cluster_size': 9, 'min_samples': 13} ===
=== MEASURES 0.003003835678100586s, {'OS': -329.55958624602675} ===
=== ALGO 16.64020538330078s, SuccessRun, {'threshold': 0.7559819902576701, 'branching_factor': 29, 'n_clusters': 3} ===
=== MEASURES 0.0016865730285644531s, {'OS': -139.30066395012994} ===
=== ALGO 0.3726999759674072s, SuccessRun, {'name': 'KMeans', 'n_clusters': 13, 'max_iter': 100, 'tol': 0.0001

/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 0.003540515899658203s, {'OS': -15516.400379752058} ===
=== MEASURES 0.0027387142181396484s, {'OS': -147.77474291163207} ===
=== ALGO 2.5344090461730957s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 13, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 11.672528743743896s, FailedRun, {'bandwidth': 0.3409627621532152, 'max_iter': 199, 'tol': 8.132138421725572e-05} ===
=== ALGO 2.6706600189208984s, FailedRun, {'name': 'DBSCAN', 'eps': 0.28830701151636184, 'eps_sq': 0.0831209328894956, 'min_samples': 12} ===
=== ALGO 16.69850254058838s, FailedRun, {'min_cluster_size': 11, 'min_samples': 15} ===
=== MEASURES 0.0016121864318847656s, {'OS': -188.58188292789353} ===
=== ALGO 26.520700693130493s, SuccessRun, {'threshold': 0.5725311769789511, 'branching_factor': 52, 'n_clusters': 11} ===
=== MEASURES 0.0014367103576660156s, {'OS': -121.26645673755615} ===
=== ALGO 0.31726932525634766s, SuccessRun, {'name': 'KMeans', 'n_clusters': 14, 'max_iter': 100, 'tol': 0.

KeyboardInterrupt: 

In [23]:
timeit.repeat(stmt="gamac_test('OpenML-waveform-5000-orig.csv', 'OS')", setup=setup, \
              repeat=5, \
              number=1, globals=None)

used data: OpenML-waveform-5000-orig.csv
used measures: [<Internal.OS: ('os', <function os at 0x71533a0b27a0>)>]


/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 0.002316713333129883s, {'OS': -16307.284231436868} ===
=== MEASURES 0.0023412704467773438s, {'OS': -155.71106853299491} ===
=== ALGO 2.3464972972869873s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 12, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 11.29454493522644s, FailedRun, {'bandwidth': 0.6162914221014213, 'max_iter': 63, 'tol': 6.125179332893423e-05} ===
=== ALGO 2.8490660190582275s, FailedRun, {'name': 'DBSCAN', 'eps': 0.7685886911534413, 'eps_sq': 0.59072857616896, 'min_samples': 11} ===
=== ALGO 2.2993874549865723s, FailedRun, {'min_cluster_size': 15, 'min_samples': 5} ===
=== MEASURES 0.009718656539916992s, {'OS': -339.42323001900616} ===
=== ALGO 36.242446422576904s, SuccessRun, {'threshold': 0.8049541286296061, 'branching_factor': 76, 'n_clusters': 3} ===
=== MEASURES 0.0023436546325683594s, {'OS': -146.789161732049} ===
=== ALGO 0.21326422691345215s, SuccessRun, {'name': 'KMeans', 'n_clusters': 11, 'max_iter': 100, 'tol': 0.0001, 'ra

/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 0.0015938282012939453s, {'OS': -16204.226856664547} ===
=== MEASURES 0.001306772232055664s, {'OS': -306.56965106539513} ===
=== ALGO 0.1612715721130371s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 4, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 11.288673162460327s, FailedRun, {'bandwidth': 0.7900990298313275, 'max_iter': 158, 'tol': 8.721156640023229e-05} ===
=== ALGO 2.5810041427612305s, FailedRun, {'name': 'DBSCAN', 'eps': 0.25439733725153046, 'eps_sq': 0.06471800520066893, 'min_samples': 13} ===
=== ALGO 2.0186967849731445s, FailedRun, {'min_cluster_size': 14, 'min_samples': 9} ===
=== MEASURES 0.0014731884002685547s, {'OS': -246.81632353020527} ===
=== ALGO 17.30513882637024s, SuccessRun, {'threshold': 0.5166420304755607, 'branching_factor': 31, 'n_clusters': 7} ===
=== MEASURES 0.0012569427490234375s, {'OS': -198.31889584816037} ===
=== ALGO 0.15683269500732422s, SuccessRun, {'name': 'KMeans', 'n_clusters': 8, 'max_iter': 100, 'tol': 0.000

...

/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 0.002591371536254883s, {'OS': -12139.982082097873} ===
=== MEASURES 0.0025339126586914062s, {'OS': -138.1656844063619} ===
=== ALGO 2.7991130352020264s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 14, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 11.362723588943481s, FailedRun, {'bandwidth': 0.2805186549529576, 'max_iter': 173, 'tol': 1.5289079322576636e-05} ===
=== ALGO 2.6754486560821533s, FailedRun, {'name': 'DBSCAN', 'eps': 0.7879261729009553, 'eps_sq': 0.6208276539423462, 'min_samples': 10} ===
=== ALGO 14.059988975524902s, FailedRun, {'min_cluster_size': 9, 'min_samples': 14} ===
=== MEASURES 0.005701541900634766s, {'OS': -282.27043217137015} ===
=== ALGO 12.36892819404602s, SuccessRun, {'threshold': 0.34450969226080796, 'branching_factor': 15, 'n_clusters': 8} ===
=== MEASURES 0.002652883529663086s, {'OS': -239.78102967955883} ===
=== ALGO 0.14460301399230957s, SuccessRun, {'name': 'KMeans', 'n_clusters': 5, 'max_iter': 100, 'tol': 0.0001,

/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 0.002256631851196289s, {'OS': -16900.97313375807} ===
=== MEASURES 0.0015850067138671875s, {'OS': -215.36824061444747} ===
=== ALGO 1.0638132095336914s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 8, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 11.753659009933472s, FailedRun, {'bandwidth': 0.3286690733856885, 'max_iter': 239, 'tol': 7.56922932123922e-05} ===
=== ALGO 2.6187145709991455s, FailedRun, {'name': 'DBSCAN', 'eps': 1.0166849033495644, 'eps_sq': 1.0336481926989132, 'min_samples': 13} ===
=== ALGO 15.701984643936157s, FailedRun, {'min_cluster_size': 7, 'min_samples': 15} ===
=== MEASURES 0.003228425979614258s, {'OS': -155.37860530538947} ===
=== ALGO 17.972689390182495s, SuccessRun, {'threshold': 0.38009617157720177, 'branching_factor': 31, 'n_clusters': 12} ===
=== MEASURES 0.0034241676330566406s, {'OS': -136.82736185654153} ===
=== ALGO 0.2688877582550049s, SuccessRun, {'name': 'KMeans', 'n_clusters': 12, 'max_iter': 100, 'tol': 0.0001,

[238.4316984219986,
 210.65451918700273,
 144.48539974500454,
 224.95286066500557,
 164.7794927759969]

In [24]:
from os import environ

In [25]:
environ["DATA"] = "OpenML_100-plants-margin_orig.csv"

In [26]:
environ["TARGET_MEASURES"] = "SYM"

In [27]:
!python3 test.py

used data: OpenML_100-plants-margin_orig.csv
used measures: [<Internal.SYM: ('sym', <function sym at 0x70eea6232e80>)>]
/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
=== MEASURES 0.08131718635559082s, {'SYM': 9.530136969134848e-06} ===
=== MEASURES 0.012997150421142578s, {'SYM': 0.003249723507943482} ===
=== ALGO 1.4496533870697021s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 11, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 2.3518483638763428s, FailedRun, {'bandwidth': 0.7031660056136334, 'max_iter': 204, 'tol': 5.910501123393775e-05} ===
=== ALGO 0.8483753204345703s, FailedRun, {'name': 'DBSCAN', 'eps': 1.1101444010854749,

In [32]:
!py-spy --version

py-spy 0.4.0


In [33]:
!py-spy record -o profile.svg -- python test.py

py-spy> Sampling process 100 times a second. Press Control-C to exit.

used data: OpenML_100-plants-margin_orig.csv
used measures: [<Internal.SYM: ('sym', <function sym at 0x76f58824ee80>)>]
/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
=== MEASURES 0.1440131664276123s, {'SYM': 8.293453616258781e-06} ===
=== MEASURES 0.014296293258666992s, {'SYM': 0.002635717894039268} ===
=== ALGO 1.4691247940063477s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 9, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 5.3782289028167725s, FailedRun, {'bandwidth': 0.8139570185518666, 'max_iter': 238, 'tol': 5.517167658182183e-05} ===
=== ALGO 2.102617

In [37]:
!py-spy top -- python3 test.py


























Total Samples 10
GIL: 50.00%, Active: 60.00%, Threads: 1

  %Own   %Total  OwnTime  TotalTime  Function (filename)                        
 10.00%  10.00%   0.010s    0.010s   _collect_parameters (typing.py)
 10.00%  60.00%   0.010s    0.060s   _call_with_frames_removed (<frozen importli
 10.00%  10.00%   0.010s    0.010s   get_data (<frozen importlib._bootstrap_exte
 10.00%  10.00%   0.010s    0.010s   _path_join (<frozen importlib._bootstrap_ex
 10.00%  10.00%   0.010s    0.010s   __new__ (enum.py)
 10.00%  60.00%   0.010s    0.060s   _load_unlocked (<frozen importlib._bootstra
  0.00%  10.00%   0.000s    0.010s   convert_class (enum.py)
  0.00%  10.00%   0.000s    0.010s   <module> (ast.py)
  0.00%  30.00%   0.000s    0.030s   exec_module (<frozen importlib._bootstrap>)
  0.00%  30.00%   0.000s    0.030s   addsitepackages (<frozen site>)
  0.00%  30.00%   0.000s    0.030s   addsitedir (<frozen site>)
  0.00%  30.00%   0.000s    0.030s   venv (<frozen site>)
